# Part 9 — Predict the noise: the loss that trains Stable Diffusion

_Rigorous Courses · Diffusion Models — Part 9 of 12_

**Rewrite the posterior mean in terms of the noise, and the whole training objective collapses to one mean-squared error**

In this notebook you verify that the reparameterization from the lesson is exact algebra (to $10^{-10}$), recompute the hand-worked example, and then do something remarkable: build a complete, working diffusion model with **no neural network at all** — because for a tiny two-point dataset, the best possible noise guess can be written down as a formula. Algorithm 2 running on that formula turns pure noise into data before your eyes.

---

This notebook accompanies the lesson. Run cells top to bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)

## The reparameterization is exact, not approximate

The lesson derived two forms of the posterior mean and claimed they are the *same function* written two ways:

- the $x_0$-form (part 7): $\tilde\mu_t = \dfrac{\sqrt{\bar\alpha_{t-1}}\,\beta_t\,x_0 + \sqrt{\alpha_t}\,(1-\bar\alpha_{t-1})\,x_t}{1-\bar\alpha_t}$
- the $\epsilon$-form (this part): $\tilde\mu_t = \dfrac{1}{\sqrt{\alpha_t}}\left(x_t - \dfrac{\beta_t}{\sqrt{1-\bar\alpha_t}}\,\epsilon\right)$

connected by the one-line sampler $x_t = \sqrt{\bar\alpha_t}\,x_0 + \sqrt{1-\bar\alpha_t}\,\epsilon$. The loss this leads to is

$$L_{\mathrm{simple}} = \mathbb{E}\,\big\|\epsilon - \epsilon_\theta(x_t, t)\big\|^2.$$

If the algebra in the lesson is right, the two mean forms must agree to machine precision on *any* inputs — not approximately, not on average. Let's check that first.

### Step 1 — Build the noise schedule

Same recipe as part 6: $T = 200$ steps, $\beta_t$ rising linearly from $10^{-4}$ to $0.02$. We also precompute $\bar\alpha_{t-1}$ (with the convention $\bar\alpha_0 = 1$), which the posterior formulas need. Arrays are 0-indexed, so the math's step $t$ lives at index `t - 1`.

In [ ]:
T = 200
betas = np.linspace(1e-4, 0.02, T)
alphas = 1.0 - betas
abar = np.cumprod(alphas)
abar_prev = np.concatenate(([1.0], abar[:-1]))   # abar_prev[t-1] holds abar_{t-1}, with abar_0 = 1

print(f"T = {T}")
print(f"beta_1 = {betas[0]:.6f}   beta_T = {betas[-1]:.6f}")
print(f"abar_1 = {abar[0]:.6f}   abar_T = {abar[-1]:.6f}")

assert np.all(np.diff(abar) < 0), "abar must strictly decrease"
assert np.all((abar > 0) & (abar < 1)), "abar must stay inside (0, 1)"


### Step 2 — Check the two forms of the posterior mean agree to machine precision

We draw one hundred thousand random combinations of clean point, noise, and timestep, build $x_t$ with the one-line sampler, then compute $\tilde\mu_t$ both ways. The largest disagreement across all of them should be at the level of floating-point rounding — about $10^{-16}$, and certainly below $10^{-10}$. That is what "exact algebra" means numerically.

In [ ]:
n_check = 100000
x0 = rng.normal(0.0, 2.0, size=n_check)
eps = rng.standard_normal(n_check)
t = rng.integers(1, T + 1, size=n_check)

b_t = betas[t - 1]
a_t = alphas[t - 1]
ab_t = abar[t - 1]
ab_prev = abar_prev[t - 1]

xt = np.sqrt(ab_t) * x0 + np.sqrt(1.0 - ab_t) * eps

mu_from_x0 = (np.sqrt(ab_prev) * b_t * x0 + np.sqrt(a_t) * (1.0 - ab_prev) * xt) / (1.0 - ab_t)
mu_from_eps = (xt - b_t / np.sqrt(1.0 - ab_t) * eps) / np.sqrt(a_t)

max_gap = np.max(np.abs(mu_from_x0 - mu_from_eps))

print(f"largest |x0-form - eps-form| over {n_check} random cases: {max_gap:.2e}")

assert max_gap < 1e-10, "the two forms must agree to machine precision"


### Step 3 — Verify the hand-worked example ($T = 3$)

The lesson's worked example used $\beta = (0.1, 0.2, 0.3)$, clean point $x_0 = 2$, true noise $\epsilon = 0.5$ at $t = 2$, and a network guess $\hat\epsilon = 0.3$. On paper we found: $x_2 = 1.9616$, per-sample loss $0.04$, implied clean point $\hat{x}_0 = 2.1247$, network mean $\mu_\theta = 2.0664$, true mean $\tilde\mu_2 = 1.9819$. Let the machine redo every line.

In [ ]:
betas3 = np.array([0.1, 0.2, 0.3])
alphas3 = 1.0 - betas3
abar3 = np.cumprod(alphas3)

x0_hand = 2.0
eps_hand = 0.5
eps_guess = 0.3

x2 = np.sqrt(abar3[1]) * x0_hand + np.sqrt(1.0 - abar3[1]) * eps_hand
loss_sample = (eps_hand - eps_guess) ** 2
x0_implied = (x2 - np.sqrt(1.0 - abar3[1]) * eps_guess) / np.sqrt(abar3[1])
mu_theta = (x2 - betas3[1] / np.sqrt(1.0 - abar3[1]) * eps_guess) / np.sqrt(alphas3[1])
mu_tilde = (x2 - betas3[1] / np.sqrt(1.0 - abar3[1]) * eps_hand) / np.sqrt(alphas3[1])
gap_direct = mu_theta - mu_tilde
gap_formula = betas3[1] / (np.sqrt(alphas3[1]) * np.sqrt(1.0 - abar3[1])) * (eps_hand - eps_guess)

print(f"x2          = {x2:.4f}   (hand: 1.9616)")
print(f"loss sample = {loss_sample:.4f}   (hand: 0.0400)")
print(f"x0 implied  = {x0_implied:.4f}   (hand: 2.1247)")
print(f"mu_theta    = {mu_theta:.4f}   (hand: 2.0664)")
print(f"mu_tilde    = {mu_tilde:.4f}   (hand: 1.9819)")
print(f"mean gap    = {gap_direct:.4f}   error-propagation formula: {gap_formula:.4f}")

assert abs(x2 - 1.9616) < 5e-4
assert abs(loss_sample - 0.04) < 1e-12
assert abs(x0_implied - 2.1247) < 5e-4
assert abs(mu_theta - 2.0664) < 5e-4
assert abs(mu_tilde - 1.9819) < 5e-4
assert abs(gap_direct - gap_formula) < 1e-12

## The exact best noise guess for a two-point world

Now the centerpiece. Take the simplest interesting dataset: the clean point is $x_0 = +2$ or $x_0 = -2$, each with probability $\tfrac12$. For this world we can write the *best possible* noise-guesser as a formula — no network, no training. Write $s = \sqrt{\bar\alpha_t}$ and $v = 1 - \bar\alpha_t$, so $q(x_t \mid x_0) = \mathcal{N}(s\,x_0,\ v)$. The derivation, one move at a time:

1. **Bayes over the two candidates** (part 2): given $x_t$, the odds for $+2$ against $-2$ are the ratio of the two Gaussian densities, $\exp\!\big(-\tfrac{(x_t - 2s)^2}{2v}\big) \big/ \exp\!\big(-\tfrac{(x_t + 2s)^2}{2v}\big)$.
2. **Expand the squares.** The $x_t^2$ and $(2s)^2$ pieces cancel between the two exponents, leaving odds $= \exp\!\big(\tfrac{4 s x_t}{v}\big)$: the further right you are, the more the evidence favors $+2$, exponentially.
3. **Best clean-point guess** is the probability-weighted average: $\mathbb{E}[x_0 \mid x_t] = 2\,p_+ - 2\,(1 - p_+)$ where $p_+$ comes from the odds. Simplifying with $2\sigma(a) - 1 = \tanh(a/2)$ gives
$$\mathbb{E}[x_0 \mid x_t] = 2\tanh\!\left(\frac{2 s\, x_t}{v}\right).$$
4. **Best noise guess** follows from the inversion identity read backwards: $\epsilon^*(x_t, t) = \dfrac{x_t - s\,\mathbb{E}[x_0 \mid x_t]}{\sqrt{v}}$.

This $\epsilon^*$ is exactly the function that minimizes $L_{\mathrm{simple}}$ — it is what a perfectly trained, infinitely flexible network would converge to. Part 10's neural network is nothing more than a stand-in for this formula in worlds (like images) where we cannot write it down.

### Step 4 — Implement the exact predictor

Two small functions: the best clean-point guess (the $\tanh$ formula) and the best noise guess built from it. A first sanity probe: at $x_t = 0$ the evidence is perfectly balanced, so the best clean-point guess is $0$ and the best noise guess is $0$ too.

In [ ]:
def x0_best_guess(xt, ab_t):
    # E[x0 | x_t] for data that is +2 or -2 with equal odds (Bayes posterior, part 2)
    s = np.sqrt(ab_t)
    v = 1.0 - ab_t
    return 2.0 * np.tanh(2.0 * s * xt / v)

def eps_best_guess(xt, ab_t):
    # best noise guess implied by the best x0 guess (the inversion identity, read backwards)
    s = np.sqrt(ab_t)
    v = 1.0 - ab_t
    return (xt - s * x0_best_guess(xt, ab_t)) / np.sqrt(v)

probe = eps_best_guess(np.array([0.0]), abar[99])

print(f"eps* at xt = 0, t = 100: {probe[0]:.6f}   (symmetry says it must be 0)")

assert abs(probe[0]) < 1e-12

### Step 5 — Fit a binned-average predictor from simulated pairs

If $\epsilon^*$ really is the conditional average $\mathbb{E}[\epsilon \mid x_t, t]$, we can rediscover it *from data alone*: simulate many $(x_t, \epsilon)$ pairs at a fixed $t$, group them into narrow bins of $x_t$, and average $\epsilon$ inside each bin. No Bayes, no $\tanh$ — pure counting. This is also exactly what a regression model trained on $L_{\mathrm{simple}}$ is trying to learn. We do it at three timesteps: early ($t = 20$), middle ($t = 100$), late ($t = 180$).

In [ ]:
def binned_eps(t_step, n_pairs, edges):
    # simulate (x_t, eps) pairs at one timestep, then average eps inside each x_t bin
    x0_draw = rng.choice(np.array([-2.0, 2.0]), size=n_pairs)
    eps_draw = rng.standard_normal(n_pairs)
    ab = abar[t_step - 1]
    xt_draw = np.sqrt(ab) * x0_draw + np.sqrt(1.0 - ab) * eps_draw
    idx = np.digitize(xt_draw, edges) - 1
    inside = (idx >= 0) & (idx < len(edges) - 1)
    n_bins = len(edges) - 1
    counts = np.bincount(idx[inside], minlength=n_bins)
    safe = np.maximum(counts, 1)
    mean_eps = np.bincount(idx[inside], weights=eps_draw[inside], minlength=n_bins) / safe
    mean_xt = np.bincount(idx[inside], weights=xt_draw[inside], minlength=n_bins) / safe
    return mean_eps, mean_xt, counts

t_show = [20, 100, 180]
edges = np.arange(-4.0, 4.0001, 0.1)
fits = {}
for t_step in t_show:
    fits[t_step] = binned_eps(t_step, 400000, edges)

sizes = [int(np.sum(fits[t_step][2])) for t_step in t_show]

print(f"timesteps fitted: {t_show}")
print(f"pairs landing inside the binned range: {sizes}")

### Step 6 — Overlay: the formula and the data agree

Solid line: the exact $\epsilon^*$ formula. Dots: the binned averages, one dot per well-populated bin. If the derivation is right, the dots must sit on the line everywhere the data reaches — and they do. (Where a bin holds too few samples its average is noisy, so we compare only bins with at least 3,000 samples.)

In [ ]:
grid = np.linspace(-4.0, 4.0, 400)
fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharey=True)
worst = 0.0
for ax, t_step in zip(axes, t_show):
    mean_eps, mean_xt, counts = fits[t_step]
    good = counts >= 3000
    exact_at_bins = eps_best_guess(mean_xt[good], abar[t_step - 1])
    gap = np.max(np.abs(mean_eps[good] - exact_at_bins))
    worst = max(worst, gap)
    ax.plot(grid, eps_best_guess(grid, abar[t_step - 1]), color="#4ea1ff", label="exact eps*")
    ax.plot(mean_xt[good], mean_eps[good], "o", ms=4, color="#ff7b72", label="binned average")
    ax.set_title(f"t = {t_step}   (worst gap {gap:.3f})")
    ax.set_xlabel("x_t")
axes[0].set_ylabel("noise guess")
axes[0].legend()
plt.suptitle("The best noise guess: formula vs pure counting")
plt.tight_layout()
plt.show()

print(f"worst formula-vs-data gap over all well-sampled bins: {worst:.4f}")

assert worst < 0.1, "binned averages must reproduce the exact predictor where data is plentiful"


## A complete diffusion model with no neural network

Everything is now on the table: a noise schedule, the exact best noise-guesser $\epsilon^*$, and Algorithm 2. Run the algorithm with $\epsilon^*$ standing where $\epsilon_\theta$ would stand:

$$x_{t-1} = \frac{1}{\sqrt{\alpha_t}}\left(x_t - \frac{\beta_t}{\sqrt{1-\bar\alpha_t}}\,\epsilon^*(x_t, t)\right) + \sigma_t z, \qquad \sigma_t^2 = \tilde\beta_t,$$

with fresh $z$ at every step except the last, where $z = 0$. If the theory of parts 6-9 is sound, 4,000 points of pure standard noise should reorganize themselves into the two-point data distribution — peaks at $-2$ and $+2$ — with zero learned parameters involved.

### Step 7 — Run Algorithm 2 from pure noise

We keep snapshots along the way (at $t = 200, 150, 100, 50, 25, 0$) so we can watch the two peaks condense out of the static.

In [ ]:
n_samples = 4000
xt_gen = rng.standard_normal(n_samples)

snap_at = {200, 150, 100, 50, 25, 0}
snaps = {200: xt_gen.copy()}

for t_step in range(T, 0, -1):
    b = betas[t_step - 1]
    a = alphas[t_step - 1]
    ab = abar[t_step - 1]
    ab_pr = abar_prev[t_step - 1]
    eps_pred = eps_best_guess(xt_gen, ab)
    mean = (xt_gen - b / np.sqrt(1.0 - ab) * eps_pred) / np.sqrt(a)
    beta_tilde = (1.0 - ab_pr) * b / (1.0 - ab)
    if t_step > 1:
        z = rng.standard_normal(n_samples)
    else:
        z = np.zeros(n_samples)
    xt_gen = mean + np.sqrt(beta_tilde) * z
    if t_step - 1 in snap_at:
        snaps[t_step - 1] = xt_gen.copy()

print(f"walked {T} reverse steps for {n_samples} samples")
print(f"final sample range: [{xt_gen.min():.2f}, {xt_gen.max():.2f}]")

### Step 8 — Watch the two peaks form, then check them

Left to right: the same 4,000 points at $t = 200$ (pure noise) down to $t = 0$ (finished samples). The single bell splits and sharpens into two spikes at $\pm 2$. The asserts then check the finished histogram quantitatively: each side's peak must sit within $0.15$ of its target, and at least 90% of samples must land within $0.5$ of a mode.

In [ ]:
snap_order = [200, 150, 100, 50, 25, 0]
fig, axes = plt.subplots(1, 6, figsize=(15, 2.8), sharey=True)
for ax, t_step in zip(axes, snap_order):
    ax.hist(snaps[t_step], bins=60, range=(-4, 4), color="#4ea1ff")
    ax.set_title(f"t = {t_step}")
    ax.set_xlabel("x")
axes[0].set_ylabel("count")
plt.suptitle("Algorithm 2 with the exact predictor: noise condensing into data")
plt.tight_layout()
plt.show()

counts, bin_edges = np.histogram(snaps[0], bins=np.arange(-3.0, 3.001, 0.05))
centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
neg_side = centers < 0
pos_side = centers > 0
peak_neg = centers[neg_side][np.argmax(counts[neg_side])]
peak_pos = centers[pos_side][np.argmax(counts[pos_side])]
dist_to_mode = np.minimum(np.abs(snaps[0] - 2.0), np.abs(snaps[0] + 2.0))
frac_near = np.mean(dist_to_mode < 0.5)

print(f"negative-side peak at {peak_neg:.3f}   (target -2)")
print(f"positive-side peak at {peak_pos:.3f}   (target +2)")
print(f"fraction of samples within 0.5 of a mode: {frac_near:.3f}")

assert abs(peak_neg + 2.0) < 0.15, "negative peak must sit near -2"
assert abs(peak_pos - 2.0) < 0.15, "positive peak must sit near +2"
assert frac_near > 0.9, "samples must concentrate at the two modes"


This is worth pausing on. **You have watched a complete, working diffusion model** — schedule, noise-prediction, Algorithm 2, the $z = 0$ ending — generate data from pure noise. The algorithm is real. The only ingredient that will change in part 10 is where the noise guess comes from: a formula here, a trained network there.

### Step 9 — The loss floor: why some timesteps are harder

Even the *best possible* predictor cannot drive $\|\epsilon - \epsilon^*\|^2$ to zero: whenever the noisy point could plausibly have come from either mode, part of the noise is fundamentally unknowable, and averaging over the possibilities leaves a remainder. We measure that irreducible floor $\mathbb{E}\big[(\epsilon - \epsilon^*(x_t, t))^2\big]$ across $t$. Expect: near zero at small $t$ (the mode is obvious, so the noise is fully recoverable), growing as noise drowns the signal and the two modes blur together. When part 10's training loss plateaus above zero, this curve is the reason — the plateau is the floor, not a failure.

In [ ]:
ts_grid = np.arange(1, T + 1, 5)
floor = []
n_mc = 20000
for t_step in ts_grid:
    x0_draw = rng.choice(np.array([-2.0, 2.0]), size=n_mc)
    eps_draw = rng.standard_normal(n_mc)
    ab = abar[t_step - 1]
    xt_draw = np.sqrt(ab) * x0_draw + np.sqrt(1.0 - ab) * eps_draw
    resid = eps_draw - eps_best_guess(xt_draw, ab)
    floor.append(np.mean(resid ** 2))
floor = np.array(floor)

plt.figure(figsize=(7, 4))
plt.plot(ts_grid, floor, color="#4ea1ff")
plt.xlabel("timestep t")
plt.ylabel("irreducible squared error")
plt.title("Loss floor of the best possible noise guess vs timestep")
plt.show()

print(f"floor at t = {ts_grid[0]}: {floor[0]:.6f}")
print(f"floor at t = {ts_grid[-1]}: {floor[-1]:.4f}")
print(f"largest floor: {floor.max():.4f}")

assert floor[0] < 1e-3, "at tiny noise the mode is obvious and the floor is ~0"
assert floor[-1] > 0.1, "at heavy noise part of the noise is unknowable"
assert floor.max() < 1.0, "the floor can never exceed the variance of eps itself"


## Practice

The same problems as the lesson. Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** At $t = 2$ in the running schedule ($\bar\alpha_2 = 0.72$) you observe $x_2 = 1.9616$, and you are told the noise that was used was $\epsilon = 0.5$. Recover the clean point $x_0$.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Inversion formula: $x_0 = (x_2 - \sqrt{1-\bar\alpha_2}\,\epsilon)/\sqrt{\bar\alpha_2}$ — solving the one-line sampler for $x_0$ (subtract, then divide) is exact.
- Constants: $\sqrt{0.28} = 0.5292$, $\sqrt{0.72} = 0.8485$.
- Numerator: $1.9616 - 0.5292 \times 0.5 = 1.6971$. Divide: $1.6971 / 0.8485 = 2.0000$.

```python
x0_rec = (1.9616 - np.sqrt(0.28) * 0.5) / np.sqrt(0.72)
print(f"recovered x0 = {x0_rec:.4f}")
```

**Answer:** $x_0 = 2.0$ — the inversion recovers the clean point exactly.

</details>

**Problem 2.** Reproduce the key collection step of the substitution: show that $\frac{\beta_t}{\sqrt{\alpha_t}} + \sqrt{\alpha_t}(1-\bar\alpha_{t-1}) = \frac{1-\bar\alpha_t}{\sqrt{\alpha_t}}$.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Common denominator $\sqrt{\alpha_t}$: the sum becomes $\frac{\beta_t + \alpha_t(1-\bar\alpha_{t-1})}{\sqrt{\alpha_t}}$ (the second term picks up $\alpha_t$ upstairs because $\sqrt{\alpha_t}\cdot\sqrt{\alpha_t} = \alpha_t$).
- Distribute: top $= \beta_t + \alpha_t - \alpha_t\bar\alpha_{t-1}$.
- Product identity $\alpha_t\bar\alpha_{t-1} = \bar\alpha_t$: top $= \beta_t + \alpha_t - \bar\alpha_t$.
- $\beta_t + \alpha_t = 1$ by the definition $\alpha_t = 1-\beta_t$: top $= 1 - \bar\alpha_t$. Done.

**Answer:** the identity holds; divided by $1-\bar\alpha_t$ it is exactly why the $x_t$ coefficient of $\tilde\mu_t$ collapses to $1/\sqrt{\alpha_t}$.

</details>

**Problem 3.** Compute the Eq. 12 weight $w_2 = \frac{\beta_2^2}{2\sigma_2^2\alpha_2(1-\bar\alpha_2)}$ for the running schedule with $\sigma_2^2 = \tilde\beta_2$, then verify the shortcut $w_t = \frac{\beta_t}{2\alpha_t(1-\bar\alpha_{t-1})}$ gives the same number.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- $\tilde\beta_2 = \frac{(1-\bar\alpha_1)\beta_2}{1-\bar\alpha_2} = \frac{0.1 \times 0.2}{0.28} = 0.0714$.
- Numerator: $\beta_2^2 = 0.04$. Denominator: $2 \times 0.0714 \times 0.8 \times 0.28 = 0.032$. So $w_2 = 0.04/0.032 = 1.25$.
- Shortcut: substituting $\sigma_t^2 = \tilde\beta_t$ cancels one $\beta_t$ and the whole $1-\bar\alpha_t$, leaving $w_t = \frac{\beta_t}{2\alpha_t(1-\bar\alpha_{t-1})} = \frac{0.2}{2\times 0.8\times 0.1} = 1.25$. Same number.

```python
beta_tilde_2 = (1 - 0.9) * 0.2 / (1 - 0.72)
w2_full = 0.2 ** 2 / (2 * beta_tilde_2 * 0.8 * (1 - 0.72))
w2_short = 0.2 / (2 * 0.8 * (1 - 0.9))
print(f"w2 full = {w2_full:.4f}   w2 shortcut = {w2_short:.4f}")
```

**Answer:** $w_2 = 1.25$ by both routes.

</details>

**Problem 4.** Trace Algorithm 1 by hand with $T = 2$, $\beta = (0.1, 0.2)$ (so $\bar\alpha = (0.9, 0.72)$). The draws are $x_0 = 1$, $t = 2$, $\epsilon = -1$, and the network currently outputs $\epsilon_\theta = -0.4$. Compute the noisy input $x_2$, the per-sample loss, and say which direction the update pushes the network's output.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- One-line sampler: $x_2 = \sqrt{0.72}\times 1 + \sqrt{0.28}\times(-1) = 0.8485 - 0.5292 = 0.3194$.
- Per-sample loss: $(-1 - (-0.4))^2 = (-0.6)^2 = 0.36$.
- Squared error always pushes the output toward the target, so the gradient step nudges $\epsilon_\theta(0.3194, 2)$ toward $-1$ (more negative).

```python
x2_trace = np.sqrt(0.72) * 1 + np.sqrt(0.28) * (-1)
loss_trace = (-1 - (-0.4)) ** 2
print(f"x2 = {x2_trace:.4f}   loss = {loss_trace:.4f}")
```

**Answer:** $x_2 = 0.3194$, loss $= 0.36$, update pushes the output toward $-1$.

</details>

**Problem 5.** Trace Algorithm 2 by hand with $T = 2$, $\beta = (0.1, 0.2)$, $\bar\alpha = (0.9, 0.72)$, using $\sigma_t = \sqrt{\beta_t}$. The draws are $x_2 = 0.9$, then $z = -1$ at step $t = 2$. The network returns $\epsilon_\theta(x_2, 2) = 0.5$ and later $\epsilon_\theta(x_1, 1) = 0.2$. Compute $x_1$ and the final output $x_0$ to 4 decimals.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- $t = 2$ mean: $\frac{1}{\sqrt{0.8}}\big(0.9 - \frac{0.2}{\sqrt{0.28}}\times 0.5\big) = \frac{0.9 - 0.1890}{0.8944} = 0.7949$.
- Add the wobble ($t = 2 > 1$): $x_1 = 0.7949 + \sqrt{0.2}\times(-1) = 0.7949 - 0.4472 = 0.3477$.
- $t = 1$ mean: $\frac{1}{\sqrt{0.9}}\big(0.3477 - \sqrt{0.1}\times 0.2\big) = \frac{0.3477 - 0.0632}{0.9487} = 0.2999$ (note $\frac{\beta_1}{\sqrt{1-\bar\alpha_1}} = \frac{0.1}{\sqrt{0.1}} = \sqrt{0.1}$).
- Final step adds no noise ($z = 0$): $x_0 = 0.2999$.

```python
x1_trace = (0.9 - 0.2 / np.sqrt(0.28) * 0.5) / np.sqrt(0.8) + np.sqrt(0.2) * (-1)
x0_trace = (x1_trace - 0.1 / np.sqrt(0.1) * 0.2) / np.sqrt(0.9)
print(f"x1 = {x1_trace:.4f}   x0 = {x0_trace:.4f}")
```

**Answer:** $x_1 = 0.3477$, generated sample $x_0 = 0.2999$.

</details>

**Problem 6.** Explain, in a short chain of reasons, why Algorithm 2 sets $z = 0$ at the final step instead of drawing fresh noise.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Each sampling step draws from $\mathcal{N}(\mu_\theta, \sigma_t^2)$ as mean $+\ \sigma_t z$ (part 3's reparameterization).
- Noise injected at step $t$ is processed — and, if unhelpful, removed — by the remaining $t-1$ denoising steps.
- The $t = 1$ update has no later step: its result *is* the returned sample.
- Noise added there would sit in the output uncorrected; reporting the mean instead gives the single best guess, because the mean of a Gaussian minimizes expected squared error (part 2).

**Answer:** fresh noise is only useful when a later step can denoise it; on the last step you deliver the mean.

</details>

**Problem 7.** Suppose you train the network to predict the clean point $\hat{x}_0$ directly, with an equal-weight loss $\mathbb{E}\|x_0 - \hat{x}_0\|^2$ averaged uniformly over $t$. Using the inversion identity, work out what this is equivalent to in noise space, and explain what goes wrong.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- At a fixed $x_t$: $\epsilon = \frac{x_t - \sqrt{\bar\alpha_t}x_0}{\sqrt{1-\bar\alpha_t}}$ and $\hat\epsilon = \frac{x_t - \sqrt{\bar\alpha_t}\hat{x}_0}{\sqrt{1-\bar\alpha_t}}$.
- Subtract (the $x_t$ terms cancel): $\epsilon - \hat\epsilon = \frac{\sqrt{\bar\alpha_t}}{\sqrt{1-\bar\alpha_t}}(\hat{x}_0 - x_0)$.
- Square: $\|\epsilon - \hat\epsilon\|^2 = \mathrm{SNR}(t)\,\|x_0 - \hat{x}_0\|^2$ with $\mathrm{SNR}(t) = \bar\alpha_t/(1-\bar\alpha_t)$ from part 6.
- So equal weight in $x_0$-space $=$ weight $1/\mathrm{SNR}(t)$ in noise space. For the canonical schedule $\mathrm{SNR}(T) \approx 4\times 10^{-5}$: the final steps get weight of order 25,000.
- But at those steps $x_t$ is nearly pure static — the best possible $\hat{x}_0$ is roughly the dataset mean, and most of that error is irreducible.
- Gradients are therefore dominated by timesteps the network cannot improve, while the detail-forming low-noise steps barely register.

**Answer:** equal-weight $x_0$-prediction over-trains the unlearnable high-noise steps and under-trains the detail-forming low-noise steps, yielding blurry, averaged-looking samples. Systems that do predict $x_0$ (or $v$) reintroduce SNR-based weights to fix exactly this.

</details>

## Wrap-up

Verified in this notebook: the $x_0$-form and $\epsilon$-form of the posterior mean agree to $10^{-10}$ on 100,000 random cases (the reparameterization is exact algebra); every number of the lesson's hand-worked example, including the error-propagation identity; the Bayes-derived $\tanh$ predictor matches a pure counting estimate wherever data is plentiful; Algorithm 2 running on that predictor turns pure noise into the two-point data distribution with peaks within $0.15$ of $\pm 2$ — a complete diffusion model with zero learned parameters; and the irreducible loss floor that rises with $t$, which is why training losses plateau above zero. Part 10 replaces the formula with the one thing we could not write down for images — a trained neural network — and builds a real DDPM in PyTorch.